# Layer diff walkthrough

This notebook walks through what `layer-diff compare` does, one step at a time, on the three
sample PDFs in `data/samples/` (regenerate them with `layer-diff make-samples`).

**How to use a notebook:** select a cell and press **Shift+Enter** to run it and move to the next.
Cells share variables, so run them top to bottom. If VS Code asks for a kernel, pick the
`.venv` interpreter of this repo.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from layer_diff.cli import PROJECT_DIR
from layer_diff.compare import compare_pages, compare_pair
from layer_diff.config import Settings
from layer_diff.describe import LocalDescriber
from layer_diff.pdf_layers import find_unlayered_content, list_layers, render_layer
from layer_diff.pipeline import run
from layer_diff.report import overlay_image
from layer_diff.samples import make_samples

sample_dir = PROJECT_DIR / "data" / "samples"
pdfs = sorted(sample_dir.glob("label_v*.pdf")) or make_samples(sample_dir)
[p.name for p in pdfs]

## 1. What layers does each file have?

A PDF "layer" is an *optional content group*. Names are matched case-insensitively later, and
content that sits outside every layer is flagged because it would show up in every layer render.

In [ ]:
for pdf in pdfs:
    names = [layer.name for layer in list_layers(pdf)]
    print(f"{pdf.name}: layers={names}  content outside layers on pages={find_unlayered_content(pdf)}")

## 2. Render one layer from each file

Rendering turns vector artwork into pixels at a fixed resolution. Because every file is rendered
the same way, identical artwork gives identical pixels: no alignment step is needed.

In [ ]:
LAYER = "Dieline"  # try "Artwork" or "varnish" (case does not matter)
settings = Settings(dpi=100)
renders = [render_layer(pdf, LAYER, dpi=settings.dpi) for pdf in pdfs]

fig, axes = plt.subplots(1, len(renders), figsize=(4 * len(renders), 6))
for ax, render in zip(axes, renders, strict=True):
    ax.imshow(render.pages[0])
    ax.set_title(f"{render.pdf_path.name}\nmatched: {render.matched_layers}")
    ax.axis("off")
plt.show()

## 3. Measure the differences between two files

`compare_pages` converts both images to CIELAB colour space (where distance approximates how
different colours *look*), marks pixels that differ by more than `color_tolerance`, groups nearby
changes into regions, and classifies each region as added / removed / changed / moved.

**Difference score** = changed pixels ÷ pixels with ink in either file. It measures how much of
the *layer's artwork* changed, independent of page size.

In [ ]:
page_a, page_b = renders[0].pages[0], renders[1].pages[0]
diff = compare_pages(page_a, page_b, settings)
print(f"score {diff.score_pct:.1f}%   SSIM {diff.ssim:.3f}   regions: {len(diff.regions)}")
pd.DataFrame([vars(r) for r in diff.regions])

In [ ]:
plt.figure(figsize=(6, 9))
plt.imshow(overlay_image(page_a, page_b, diff))
plt.title("red = only in A, blue = only in B, amber = changed; purple box = moved")
plt.axis("off")
plt.show()

## 4. Plain-English description

The local describer turns the measurements into sentences: position on the page, size in
millimetres, colour names and movement direction. It runs offline and is deterministic.

In [ ]:
pair = compare_pair(renders[0], renders[1], settings)
text = LocalDescriber().describe(pair, settings)
print(text.summary)
for line in text.details:
    print(" -", line)

## 5. The whole thing in one call: 2..N files, several layers, a report

This is exactly what `layer-diff compare` runs. With N files, *pairwise* mode compares every pair
(N×(N−1)/2 comparisons) and the overall score is their mean; *baseline* mode compares each file
with the first one.

In [ ]:
out_dir = PROJECT_DIR / "data" / "output" / "notebook_demo"
report = run(pdfs, ["Dieline", "Artwork", "Varnish"], Settings(dpi=100), out_dir)

for layer in report["layers"]:
    names = [Path(p).name for p in layer["matrix"]["files"]]
    print(f"\n{layer['pattern']}: overall {layer['overall_score_pct']:.1f}%")
    display(pd.DataFrame(layer["matrix"]["scores"], index=names, columns=names).round(1))
print("Report written to", out_dir / "report.md")

## 6. Your own files

1. Copy production PDFs into `projects/layer_diff/data/input/` (that folder is git-ignored).
2. Point `pdfs` at them, e.g. `pdfs = sorted((PROJECT_DIR / "data" / "input").glob("*.pdf"))`.
3. Run step 1 to see the real layer names, set `LAYER`, and re-run the cells.
4. Tune: raise `dpi` (e.g. 300) for tiny text; raise `color_tolerance` if colour-managed exports
   create noise; raise `min_region_mm2` to ignore specks.

Optional richer wording via Claude (sends the images to the Claude API; needs `ANTHROPIC_API_KEY`):

```python
from layer_diff.describe import ClaudeDescriber
text = ClaudeDescriber().describe(pair, settings, images=(page_a, page_b, overlay_image(page_a, page_b, diff)))
```